# LSTM Optimisation Comparison


## 1. Load LSTM-Ready Cache


In [ ]:
# Purpose: Loads the optional LSTM-ready cache. This notebook does not scan audio folders,
# make a new split, or call Librosa MFCC extraction.
import json
import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from tensorflow.keras.layers import Dense, Dropout, Input, LSTM
from tensorflow.keras.models import Sequential

try:
    from IPython.display import display
except Exception:
    display = print

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
tf.keras.utils.set_random_seed(RANDOM_STATE)
os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)

CLASS_NAMES = {0: "bona_fide", 1: "synthetic"}
EPOCHS = 30
BATCH_SIZE = 64
THRESHOLD = 0.5


def resolve_project_root():
    explicit = os.environ.get("INTRO_AI_PROJECT_ROOT")
    if explicit:
        return Path(explicit).expanduser().resolve()
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "Model Variants").exists():
            return candidate
        if (candidate / "Training" / "Model Variants").exists():
            return candidate / "Training"
    return Path("/content/drive/MyDrive/Colab Notebooks/Education/INM701")


PROJECT_ROOT = resolve_project_root()
SHARED_CLASS_WEIGHT_PATH = PROJECT_ROOT / "outputs" / "shared" / "class_weights.json"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "lstm_optional"
TABLES_DIR = OUTPUT_DIR / "tables"
CACHE_DIR = OUTPUT_DIR / "cache"
MODELS_DIR = OUTPUT_DIR / "models"
METRICS_DIR = OUTPUT_DIR / "metrics"
FIGURES_DIR = OUTPUT_DIR / "figures"
for directory in [OUTPUT_DIR, TABLES_DIR, CACHE_DIR, MODELS_DIR, METRICS_DIR, FIGURES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


def require_file(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Required file not found: {path}. Run the analysis notebook, then 00_LSTM_Data_Preparation.ipynb."
        )
    return path


def load_shared_class_weights(path):
    with open(require_file(path), "r", encoding="utf-8") as f:
        payload = json.load(f)
    weights = payload.get("class_weights", payload)
    return {int(label): float(weight) for label, weight in weights.items()}


for required in [
    CACHE_DIR / "X_train.npy",
    CACHE_DIR / "y_train.npy",
    CACHE_DIR / "train_metadata.csv",
    CACHE_DIR / "X_validation.npy",
    CACHE_DIR / "y_validation.npy",
    CACHE_DIR / "validation_metadata.csv",
    CACHE_DIR / "feature_config.json",
    SHARED_CLASS_WEIGHT_PATH,
]:
    require_file(required)

X_train_scaled = np.load(CACHE_DIR / "X_train.npy")
y_train = np.load(CACHE_DIR / "y_train.npy")
train_meta = pd.read_csv(CACHE_DIR / "train_metadata.csv")
X_validation_scaled = np.load(CACHE_DIR / "X_validation.npy")
y_validation = np.load(CACHE_DIR / "y_validation.npy")
validation_meta = pd.read_csv(CACHE_DIR / "validation_metadata.csv")
mfcc_mean = np.load(CACHE_DIR / "mfcc_mean.npy")
mfcc_std = np.load(CACHE_DIR / "mfcc_std.npy")
with open(CACHE_DIR / "feature_config.json", "r", encoding="utf-8") as f:
    feature_config = json.load(f)
CLASS_WEIGHTS = load_shared_class_weights(SHARED_CLASS_WEIGHT_PATH)
class_weights = CLASS_WEIGHTS



print("Loaded LSTM-ready cache:", CACHE_DIR)
print("X_train shape:", X_train_scaled.shape)
print("X_validation shape:", X_validation_scaled.shape)
print("Shared class weights:", CLASS_WEIGHTS)


## 2. Compare Optimisers and Optional Confirmation


In [ ]:
# Purpose: Compares validation-only Random Search and GA outputs when they exist. It
# does not load or score the test set.
RUN_CONFIRMATION = False
OPTIMISATION_DIR = OUTPUT_DIR / "optimisation"
OPTIMISATION_DIR.mkdir(parents=True, exist_ok=True)
random_path = OPTIMISATION_DIR / "random_search_results.csv"
ga_path = OPTIMISATION_DIR / "genetic_algorithm_results.csv"
selected_path = OPTIMISATION_DIR / "selected_hyperparameters.json"


def maybe_load(path, method):
    if not path.exists():
        print(f"Missing {method} results:", path)
        return pd.DataFrame()
    df = pd.read_csv(path)
    df["method"] = method
    return df


random_df = maybe_load(random_path, "random_search")
ga_df = maybe_load(ga_path, "genetic_algorithm")
candidates = pd.concat([random_df, ga_df], ignore_index=True)
if candidates.empty:
    print("No optimisation outputs are available yet.")
else:
    candidates = candidates.sort_values("f1", ascending=False)
    display(candidates.head())
    best = candidates.iloc[0].to_dict()
    selected = {"method": best.get("method"), "config": {key: best[key] for key in ["units", "dropout", "learning_rate", "batch_size"] if key in best}}
    with open(selected_path, "w", encoding="utf-8") as f:
        json.dump(selected, f, indent=2)
    print("Saved selected hyperparameters:", selected_path)

if RUN_CONFIRMATION:
    print("Confirmation seeds should train the selected config on train+validation only.")
else:
    print("RUN_CONFIRMATION is False. Enable only after Random Search/GA results exist.")


## 3. Test Set Deliberately Unused


In [ ]:
display(pd.DataFrame([{"check": "test_metrics_computed_here", "value": False}, {"check": "uses_shared_lstm_cache", "value": str(CACHE_DIR)}]))
